In [ ]:
!pip install mujoco mediapy

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'   # rendering senza schermo, serve su Colab

import mujoco
import mediapy as media
import numpy as np

print("MuJoCo version:", mujoco.__version__)

MuJoCo version: 4.0.0


In [ ]:
xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="2 2 0.1" rgba=".9 .9 .9 1"/>
    <body name="pole" pos="0 0 1">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.6" size="0.04" rgba="0 .6 .8 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=320, width=480)

frames = []
mujoco.mj_resetData(model, data)
data.qvel[0] = 0.01   # diamo una spinta iniziale al braccio
while data.time < 4:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * 30:
        renderer.update_scene(data)
        frames.append(renderer.render())

media.show_video(frames, fps=30)

In [ ]:
import numpy as np
print(model.nq, model.nv, model.nbody)    # dimensioni dello stato e numero di corpi
print(model.body_mass)                    # mondo e asta
print(data.qpos[0] / (2 * np.pi))         # cosa misura questo numero?
print(data.xpos[1], data.geom_xpos[1])    # origine del corpo e centro della capsula

1 1 2
[0.         3.28401152]
3.1700419253490955
[0. 0. 1.] [0.26138676 0.         1.14723098]


In [ ]:
import matplotlib.pyplot as plt

xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="2 2 0.1" rgba=".9 .9 .9 1"/>
    <body name="pole" pos="0 0 1">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 0.6" size="0.04" rgba="0 .6 .8 1"/>
    </body>
  </worldbody>
  <actuator>
    <position name="servo" joint="hinge" kp="100" kv="12"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=320, width=480)

frames, t, theta = [], [], []
mujoco.mj_resetData(model, data)
data.ctrl[0] = 0.5   # angolo comandato: 0,5 rad, circa 29 gradi
while data.time < 3:
    mujoco.mj_step(model, data)
    t.append(data.time)
    theta.append(data.qpos[0])
    if len(frames) < data.time * 30:
        renderer.update_scene(data)
        frames.append(renderer.render())

media.show_video(frames, fps=30)

plt.plot(t, theta, label="angolo misurato")
plt.axhline(0.5, linestyle="--", color="gray", label="angolo comandato")
plt.xlabel("tempo [s]"); plt.ylabel("θ [rad]"); plt.legend(); plt.show()